Transformación datos bronze a capa silver. -- 07/11/20226

Este notebook tendrá las siguientes responsabilidades:
1. Leer la tabla facturacion de la capa bronze 
2. Realizar limpieza y validacion de los datos por medio de las siguientes reglas de negocio de Aguas de Manizales
3.Escribir los registros válidos en silver.facturacion_transformed y los rechazados en silver.facturacion_rechazados



 

Reglas aplicadas a los registros de facturación:

PERIODO: nulo, formato distinto a yyyyMM (mes 01-12) o periodo futuro
Suscriptor: nulo o fuera del rango 0 - 200.000  (suscriptores acutales de Aguas de Manizales)
Cliente / dirección: nombre vacío o SIN IDENTIFICAR, dirección vacía
ESTRATO: valor distinto de 1-6 (el estrato nulo se permite ya que corresponde a suscriptores NO residenciales,o viene como SIN ESTRATO)
CONCEPTO: código o descripción nula
CONSUMO: nulo, negativo o atípico (mayor a Q3 + 3·IQR)
VALOR CARGO FIJO- VALOR CONSUMO Y VALOR TOTAL ($): valores nulos o negativos, valor total faltante, negativo o distinto de la sumatoria de valor cargo fijo ($) + valor consumo ($)
DUPLICADOS: registros idénticos (se conserva uno,mismo periodo y mismo suscriptor). 

Los registros válidos van a la capa silver , tabla silver.facturacion_transformed
Los registros rechazados van a la capa silver, tabla silver.facturacion_rechazados con sus motivos para revision por parte de los analista de negocio de AGM.


In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F  
from pyspark.sql import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_pysmartdata")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_bronze_fact = spark.table(f"{catalogo}.{esquema_source}.facturacion")

# Parámetros de calidad
SUSCRIPTOR_MIN, SUSCRIPTOR_MAX = 0, 200000    ## de acuerdo al No. de suscriptores acutales de AGM
ESTRATOS_VALIDOS = [1, 2, 3, 4, 5, 6]      
TOLERANCIA_TOTAL = 1.0   # diferencia máxima (pesos) entre valor_total y cargo fijo + valor consumo
FACTOR_IQR = 3.0         # 3 = solo atípicos extremos

# Límite superior de consumo atípico (IQR sobre consumos no negativos)
q1, q3 = (df_bronze_fact
          .filter(F.col("consumo_m3") >= 0)
          .approxQuantile("consumo_m3", [0.25, 0.75], 0.001))
limite_sup_consumo = q3 + FACTOR_IQR * (q3 - q1)
print(f"Q1={q1}, Q3={q3}, límite superior consumo atípico={limite_sup_consumo} m3")

In [0]:
cols_negocio = [c for c in df_bronze_fact.columns if c != "ingestion_date"]
w_dup = Window.partitionBy(*cols_negocio).orderBy(F.col("ingestion_date").desc())

periodo_str = F.col("periodo").cast("string")
nombre_norm = F.upper(F.trim(F.col("nombre_cliente")))

reglas = {
    "PERIODO_NULO":                  F.col("periodo").isNull(),
    "PERIODO_FORMATO_INVALIDO":      F.col("periodo").isNotNull() & ~periodo_str.rlike(r"^(19|20)[0-9]{2}(0[1-9]|1[0-2])$"),
    "PERIODO_FUTURO":                F.col("periodo") > F.date_format(F.current_date(), "yyyyMM").cast("int"),
    "SUSCRIPTOR_NULO":               F.col("suscriptor").isNull(),
    "SUSCRIPTOR_FUERA_DE_RANGO":     ~F.col("suscriptor").between(SUSCRIPTOR_MIN, SUSCRIPTOR_MAX),
    "CLIENTE_NO_IDENTIFICADO":       F.col("nombre_cliente").isNull() | (nombre_norm == "") | nombre_norm.contains("SIN IDENTIFICAR"),
    "DIRECCION_VACIA":               F.col("direccion").isNull() | (F.trim(F.col("direccion")) == ""),
    "ESTRATO_INVALIDO":              F.col("estrato").isNotNull() & ~F.col("estrato").isin(ESTRATOS_VALIDOS),
    "CONCEPTO_NULO":                 F.col("codigo_concepto").isNull() | F.col("concepto").isNull(),
    "CONSUMO_NULO":                  F.col("consumo_m3").isNull(),
    "CONSUMO_NEGATIVO":              F.col("consumo_m3") < 0,
    "CONSUMO_ATIPICO":               F.col("consumo_m3") > F.lit(limite_sup_consumo),
    "CARGO_FIJO_NULO_O_NEGATIVO":    F.col("valor_cargo_fijo").isNull() | (F.col("valor_cargo_fijo") < 0),
    "VALOR_CONSUMO_NULO_O_NEGATIVO": F.col("valor_consumo").isNull() | (F.col("valor_consumo") < 0),
    "VALOR_TOTAL_FALTANTE":          F.col("valor_total").isNull(),
    "VALOR_TOTAL_NEGATIVO":          F.col("valor_total") < 0,
    "VALOR_TOTAL_INCONSISTENTE":     F.abs(F.col("valor_total") - (F.col("valor_cargo_fijo") + F.col("valor_consumo"))) > TOLERANCIA_TOTAL,
    "REGISTRO_DUPLICADO":            F.col("_rn") > 1,
}

df_validado = (
    df_bronze_fact
    .withColumn("_rn", F.row_number().over(w_dup))
    .withColumn(
        "motivos_rechazo",
        F.filter(F.array(*[F.when(cond, F.lit(nombre)) for nombre, cond in reglas.items()]),
                 lambda x: x.isNotNull())
    )
    .withColumn("es_valido", F.size("motivos_rechazo") == 0)
)

In [0]:
total_registros = df_validado.count()
print(f"Total registros bronze: {total_registros}")

display(df_validado.groupBy("es_valido").count())

# Registros por motivo de rechazo (un registro puede tener varios motivos)
display(
    df_validado
    .select(F.explode("motivos_rechazo").alias("motivo"))
    .groupBy("motivo").count()
    .withColumn("pct", F.round(F.col("count") * 100 / total_registros, 4))
    .orderBy(F.desc("count"))
)

# Advertencia (no se rechaza): estrato nulo
print("Registros con estrato nulo (advertencia):", df_validado.filter(F.col("estrato").isNull()).count())

In [0]:
tabla_silver = f"{catalogo}.{esquema_sink}.facturacion_transformed"
tabla_rechazos = f"{catalogo}.{esquema_sink}.facturacion_rechazados"

# Válidos: normalizar textos y respetar el orden de columnas de la tabla silver
df_validos = (
    df_validado
    .filter(F.col("es_valido"))
    .withColumn("nombre_cliente", F.upper(F.trim(F.col("nombre_cliente"))))
    .withColumn("direccion", F.upper(F.trim(F.col("direccion"))))
    .withColumn("concepto", F.trim(F.col("concepto")))
    .select(spark.table(tabla_silver).columns)
)

# Rechazados: se conservan con sus motivos para revisión y reproceso
df_rechazados = (
    df_validado
    .filter(~F.col("es_valido"))
    .drop("_rn", "es_valido")
    .withColumn("fecha_validacion", F.current_timestamp())
    .select(spark.table(tabla_rechazos).columns)
)

In [0]:
df_validos.write.mode("overwrite").insertInto(tabla_silver)

df_rechazados.write.mode("overwrite").insertInto(tabla_rechazos)

In [0]:
n_silver = spark.table(tabla_silver).count()
n_rech = spark.table(tabla_rechazos).count()
print(f"Bronze: {total_registros} | regisros silver válidos: {n_silver} | registros silver rechazados: {n_rech}")
assert n_silver + n_rech == total_registros, "Los conteos no cuadran"

In [0]:
df_validos.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.facturacion_transformed")

In [0]:
%sql
SELECT *
FROM catalog_pysmartdata.silver.facturacion_transformed
LIMIT 100;

In [0]:
%sql
SELECT periodo, suscriptor, nombre_cliente, consumo_m3, valor_total,
       array_join(motivos_rechazo, ', ') AS motivos, fecha_validacion
FROM catalog_pysmartdata.silver.facturacion_rechazados
ORDER BY periodo, suscriptor;

In [0]:
%sql
SELECT motivo, COUNT(*) AS registros
FROM catalog_pysmartdata.silver.facturacion_rechazados
LATERAL VIEW explode(motivos_rechazo) t AS motivo
GROUP BY motivo
ORDER BY registros DESC;